# Where to cut a regression leaf

The four yields sit at settings $1, 2, 3, 4$:

$$
2, 2, 8, 8
$$

The parent mean is $5$ and the parent sum of squares is $36$. A cut's score is the sum of the two children's sums of squares. The reduction is $36$ minus that sum. The best cut is the one with the largest reduction.

The gaps are between $1$ and $2$, between $2$ and $3$, and between $3$ and $4$. The midpoints are $3/2$, $5/2$, and $7/2$.


## Three cuts

Cut at $5/2$. Left yields $2, 2$, mean $2$, sum of squares $0$. Right yields $8, 8$, mean $8$, sum of squares $0$. Reduction $36 - 0 = 36$.

Cut at $3/2$. Left is the single value $2$, sum of squares $0$. Right is $2, 8, 8$.

$$
\text{mean} = \frac{2+8+8}{3} = 6
$$

$$
(2-6)^2 + (8-6)^2 + (8-6)^2 = 16 + 4 + 4 = 24
$$

Reduction $36 - 24 = 12$.

Cut at $7/2$ is the mirror: left $2, 2, 8$ has mean $4$ and sum of squares $24$, right is $0$. Reduction $12$ again.

The middle cut removes every squared error in the training yields. The side cuts leave $24$.


In [1]:
# Three candidate cuts. Only the middle one drives the squared error to 0.
from fractions import Fraction

settings = [1, 2, 3, 4]
yields = [2, 2, 8, 8]

def sse(values):
    if not values:
        return Fraction(0)
    mean = Fraction(sum(values), len(values))
    return sum((y - mean) ** 2 for y in values)

parent = sse(yields)
assert parent == 36

def reduction(threshold):
    left = [y for x, y in zip(settings, yields) if x <= threshold]
    right = [y for x, y in zip(settings, yields) if x > threshold]
    return parent - sse(left) - sse(right)

assert reduction(Fraction(5, 2)) == 36
assert reduction(Fraction(3, 2)) == 12
assert reduction(Fraction(7, 2)) == 12
print("reduction at 5/2", reduction(Fraction(5, 2)))
print("reduction at 3/2", reduction(Fraction(3, 2)))


reduction at 5/2 36
reduction at 3/2 12


## What the leaves say

Left of $5/2$, predict $2$. Right of $5/2$, predict $8$. A new setting of $3$ goes right and predicts $8$. A new setting of $2$ goes left and predicts $2$. Setting $2.5$ itself satisfies $\le 5/2$, so it predicts $2$. The midpoint belongs to the left branch because the rule is $\le$, and that choice does not move any training point.


In [2]:
# Predict with the winning cut. Training squared error is 0.
from fractions import Fraction

def predict(setting):
    if setting <= Fraction(5, 2):
        return 2
    return 8

yields = [2, 2, 8, 8]
predictions = [predict(x) for x in (1, 2, 3, 4)]
assert predictions == yields
assert predict(Fraction(5, 2)) == 2
assert predict(3) == 8
print("predictions", predictions)


predictions [2, 2, 8, 8]


## A pitfall

A reduction of $36$ means the two leaves fit these four numbers exactly. Two points determine a mean with no leftover. That is a perfect training fit, and it is also a leaf with no way to describe variation inside the leaf. The next lesson is what happens when one odd point creates a leaf of its own.

## What to carry forward

Split a regression node by the drop in sum of squares. The parent costs $36$. The cut at $5/2$ costs $0$. The cuts at $3/2$ and $7/2$ each cost $24$.
